# Code Exercise 1: Prim's Algorithm

In [4]:
import heapq

edge_weight = [(1, 2, 4), (2, 3, 9), (3, 4, 6), (4, 5, 3),
               (5, 3, 4), (5, 6, 2), (6, 2, 9), (6, 3, 2),
               (6, 7, 8), (7, 2, 7), (7, 5, 9), (7, 8, 8),
               (8, 4, 9), (8, 5, 9), (8, 9, 18), (9, 1, 4),
               (9, 7, 10), (9, 10, 3), (10, 1, 1), (10, 2, 5), (10, 7, 9)]

class Edge:
    def __init__(self, v1, v2, weight):
        self.u, self.v, self.w = v1, v2, weight
    def __eq__(self, another):
        return self.w == another.w
    def __lt__(self, another):
        return self.w < another.w
    def __gt__(self, another):
        return self.w > another.w
    def __le__(self, another):
        return self.w <= another.w
    def __ge__(self, another):
        return self.w >= another.w

adjlist, edge = {}, []
for i in edge_weight:
    e = Edge(i[0], i[1], i[2])
    edge.append(e)
    if e.u not in adjlist:
        adjlist[e.u] = [e]
    else:
        adjlist[e.u].append(e)
    if e.v not in adjlist:
        adjlist[e.v] = [e]
    else:
        adjlist[e.v].append(e)

vertex = list(adjlist.keys())
num_vertex = len(vertex)

start = vertex[0]
tree_node = [start]
mst, heap = [], []
for e in adjlist[start]:
    heapq.heappush(heap, e)

while len(tree_node) != num_vertex:
    e = heapq.heappop(heap)

    if e.u in tree_node and e.v in tree_node:
        continue

    n = e.v if e.v not in tree_node else e.u
    # n = e.v
    tree_node.append(n)
    mst.append(e)
    for e in adjlist[n]:
        if e.v not in tree_node:
            heapq.heappush(heap, e)

for e in mst:
    print(e.u, e.v, e.w)


10 1 1
9 1 4
1 2 4
10 7 9
7 8 8
7 5 9
5 6 2
6 3 2
3 4 6


# Code Exercise 2: Dijkstra's Algorithm

In [ ]:
import math

# lst = [[0, u], [10, v], [20, z]]  first one is the distance, second one is the node idx
class Heap:
    def __init__(self, lst):
        self.__size = len(lst)
        self.__capacity = 2 ** math.ceil(math.log(self.__size + 1, 2))
        self.__key = [None] * self.__capacity
        self.__key_pos = {}  # this is added to memorize the neighbors' positions
        for i in range(self.__size):
            self.__key[i + 1] = lst[i]
            self.__key_pos[lst[i][1]] = i + 1  # initialization of the key_position

    def __try_swap(self, i, j):
        if self.__key[j][0] < self.__key[i][0]:
            self.__key[i], self.__key[j] = self.__key[j], self.__key[i]
            self.__key_pos[self.__key[i][1]], self.__key_pos[self.__key[j][1]] = i, j
            # if the heap is changed, i.e., some keys are exchanges, update the key_pos
            return True
        else:
            return False

    def __heapify_at(self, i):
        if 2 * i > self.__size:
            return
        if 2 * i == self.__size or self.__key[2 * i][0] <= self.__key[2 * i + 1][0]:
            if self.__try_swap(i, 2 * i):
                self.__heapify_at(2 * i)
        else:
            if self.__try_swap(i, 2 * i + 1):
                self.__heapify_at(2 * i + 1)

    def heapify(self):
        for i in range(self.__size // 2, 0, -1):
            self.__heapify_at(i)

    def update(self, k):
        if k[1] in self.__key_pos:  # if the node is alreay in the heap
            i = self.__key_pos[k[1]]
            if self.__key[i][0] > k[0]:
                self.__key[i][0] = k[0]
                while i > 1 and self.__try_swap(i // 2, i):
                    i = i // 2
        else: # if not, add it as a new element
            if self.__size + 1 == self.__capacity:
                self.__key.extend([None] * self.__capacity)
                self.__capacity *= 2
            self.__size += 1
            i = self.__size
            self.__key[i] = k
            self.__key_pos[k[1]] = i
            while i > 1 and self.__try_swap(i // 2, i):
                i = i // 2

    def pop(self):
        if self.__size == 0:
            return None
        else:
            k = self.__key[1]
            if self.__size > 1:
                del self.__key_pos[self.__key[1][1]]
                self.__key[1], self.__key[self.__size] = self.__key[self.__size], None
                self.__key_pos[self.__key[1][1]] = 1
                self.__size -= 1
                self.__heapify_at(1)
            else:
                self.__size = 0
            return k

    def len(self):
        return self.__size

    def print(self):
        print('the content is', self.__key[1:self.__size + 1])


wg = [(1, 2, 4), (2, 3, 9), (3, 4, 6), (4, 5, 3),
      (5, 3, 4), (5, 6, 2), (6, 2, 9), (6, 3, 2),
      (6, 7, 8), (7, 2, 7), (7, 5, 9), (7, 8, 8),
      (8, 4, 9), (8, 5, 9), (8, 9, 18), (9, 1, 4),
      (9, 7, 10), (9, 10, 3), (10, 1, 1), (10, 2, 5), (10, 7, 9)]

vertex = set()
for u, v, w in wg:
    vertex.add(u)
    vertex.add(v)
vertex = list(vertex)

adjlist = {u: [] for u in vertex}
for u, v, w in wg:
    adjlist[u].append((v, w))
    adjlist[v].append((u, w))
print('to vertex |', end='')
for u in vertex:
    print('%4d' % u, end='')
print()
print('-' * 51)
for u in vertex:
    heap = Heap([[0, u]])
    dist = {}
    while heap.len() > 0:
        mindist, v = heap.pop()
        dist[v] = mindist
        for i, w in adjlist[v]:
            if i not in dist:
                heap.update([mindist + w, i])
    print('from %4d |' % u, end='')
    for v in vertex:
        print('%4d' % dist[v], end='')
    print()


to vertex |   1   2   3   4   5   6   7   8   9  10
---------------------------------------------------
from    1 |   0   4  13  18  15  13  10  18   4   1
from    2 |   4   0   9  14  11   9   7  15   8   5
from    3 |  13   9   0   6   4   2  10  13  17  14
from    4 |  18  14   6   0   3   5  12   9  22  19
from    5 |  15  11   4   3   0   2   9   9  19  16
from    6 |  13   9   2   5   2   0   8  11  17  14
from    7 |  10   7  10  12   9   8   0   8  10   9
from    8 |  18  15  13   9   9  11   8   0  18  17
from    9 |   4   8  17  22  19  17  10  18   0   3
from   10 |   1   5  14  19  16  14   9  17   3   0
